In [2]:
# 1. IMPORT LIBRARY

import numpy as np
import tensorflow as tf
import tensorflow_addons as tfa
import random
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout
from tf2crf import CRF, ModelWithCRFLoss
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

print("Versi TensorFlow:", tf.__version__)

Versi TensorFlow: 2.15.0


In [ ]:
# 2. PARSING DATA BIO

def parse_conll_file(file_path):
    sentences = []
    labels = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        current_sentence = []
        current_labels = []
        for line in f:
            line = line.strip()
            if not line:
                if len(current_sentence) > 0:
                    sentences.append(current_sentence)
                    labels.append(current_labels)
                current_sentence = []
                current_labels = []
            else:
                parts = line.split('\t')
                if len(parts) >= 2:
                    current_sentence.append(parts[0])
                    current_labels.append(parts[-1])
                    
    return sentences, labels

semua_sentences, semua_labels = parse_conll_file("doccanoBIO_bersih.txt")

train_sentences, test_sentences, train_labels, test_labels = train_test_split(
    semua_sentences, semua_labels, test_size=0.2, random_state=42
)

print(f"Total Seluruh Data: {len(semua_sentences)}")
print(f"Total Data Latih (Train): {len(train_sentences)}")
print(f"Total Data Uji (Test): {len(test_sentences)}")

Total Seluruh Data: 1252
Total Data Latih (Train): 1001
Total Data Uji (Test): 251


In [ ]:
from collections import Counter
from difflib import get_close_matches
import re

DAFTAR_INDUK = [
    "INDOMARET", "ALFAMART", "ALFAMIDI", "GEMILANG", "ROONA COFFEE", 
    "SUSHI TEI", "ACE HARDWARE", "HYPERMART", "MR DIY", "BREADTALK", 
    "CHATTING CAFE", "INFORMA", "MIXUE", "INDOGROSIR", "LION SUPERINDO",
    "POPEYES", "KFC", "MCDONALDS", "HOLLAND BAKERY", "NESCAFE", "MONETTA",
    "APOTEK ROXY", "FANIDA AP", "APTEK SALWA", "ROTI O", "LAWSON", "FAMILYMART",
    "CV KHASANAH SARI", "BINGXUE", "ANNIDA", "O SAVE"
]

def normalisasi_nama_toko(nama):
    n = re.sub(r'[^A-Z0-9]', '', nama.upper())
    if not n: return "TANPA_NAMA_TOKO"
    
    matches = get_close_matches(n, [re.sub(r'[^A-Z0-9]', '', induk) for induk in DAFTAR_INDUK], n=1, cutoff=0.4)
    
    if matches:
        idx = [re.sub(r'[^A-Z0-9]', '', induk) for induk in DAFTAR_INDUK].index(matches[0])
        return DAFTAR_INDUK[idx]
    
    return "TOKO_LOKAL_LAIN"

def cek_distribusi(sentences, labels):
    toko_counter = Counter()
    
    for seq, lab in zip(sentences, labels):
        nama_toko = [word.lower() for word, tag in zip(seq, lab) if tag in ['B-nama_toko', 'I-nama_toko']]
        toko_str = " ".join(nama_toko).strip().upper()
        
        if not toko_str:
            toko_str = "TANPA_NAMA_TOKO"
        else:
            toko_str = normalisasi_nama_toko(toko_str)
            
        toko_counter[toko_str] += 1
        
    print("=== DISTRIBUSI JENIS NOTA (SUDAH DIBERSIHKAN) ===")
    for toko, jumlah in toko_counter.most_common():
        print(f"- {toko}: {jumlah} nota")
        
    return toko_counter

distribusi = cek_distribusi(train_sentences, train_labels)

=== DISTRIBUSI JENIS NOTA (SUDAH DIBERSIHKAN) ===
- INDOMARET: 276 nota
- TANPA_NAMA_TOKO: 154 nota
- ALFAMART: 102 nota
- TOKO_LOKAL_LAIN: 97 nota
- ALFAMIDI: 35 nota
- GEMILANG: 32 nota
- MONETTA: 29 nota
- ROONA COFFEE: 26 nota
- ACE HARDWARE: 21 nota
- BREADTALK: 21 nota
- SUSHI TEI: 21 nota
- HYPERMART: 19 nota
- FAMILYMART: 17 nota
- CHATTING CAFE: 16 nota
- FANIDA AP: 15 nota
- HOLLAND BAKERY: 13 nota
- CV KHASANAH SARI: 10 nota
- ANNIDA: 10 nota
- NESCAFE: 9 nota
- LION SUPERINDO: 9 nota
- MR DIY: 9 nota
- INFORMA: 8 nota
- APOTEK ROXY: 8 nota
- APTEK SALWA: 8 nota
- O SAVE: 7 nota
- INDOGROSIR: 7 nota
- ROTI O: 6 nota
- LAWSON: 4 nota
- MIXUE: 4 nota
- MCDONALDS: 3 nota
- BINGXUE: 2 nota
- POPEYES: 2 nota
- KFC: 1 nota


In [ ]:
# 2.5 DATA AUGMENTATION

def augment_number_token(token, label):
    """Mengganti angka acak untuk melatih AI mengenali variasi harga/kuantitas"""
    if label in ['B-harga_satuan', 'I-harga_satuan', 'B-harga_total_barang', 'I-harga_total_barang', 'B-diskon', 'I-diskon', 'B-total_belanja', 'I-total_belanja']:
        if re.match(r'^[0-9.,]+$', token):
            length = len(token.replace('.', '').replace(',', ''))
            new_num = ''.join([str(random.randint(1, 9)) for _ in range(length)])
            return new_num
    elif label in ['B-jumlah', 'I-jumlah']:
        if token.isdigit() and len(token) <= 2:
            return str(random.randint(1, 9))
    return token

def augment_data_imbalance(sentences, labels, distribusi_toko, batas_minoritas=50):
    aug_sentences = []
    aug_labels = []
    
    for seq, lab in zip(sentences, labels):
        nama_toko = [w.lower() for w, t in zip(seq, lab) if t in ['B-nama_toko', 'I-nama_toko']]
        toko_str = " ".join(nama_toko).strip()
        if not toko_str:
            toko_str = "TANPA_NAMA_TOKO"
            
        jumlah_asli = distribusi_toko[toko_str]

        if jumlah_asli < 20:
            faktor = 5
        elif jumlah_asli < batas_minoritas:
            faktor = 3
        else:
            faktor = 1 
            
        for _ in range(faktor):
            new_seq = []
            for token, tag in zip(seq, lab):
                if random.random() < 0.3:
                    new_seq.append(augment_number_token(token, tag))
                else:
                    new_seq.append(token)
            
            aug_sentences.append(new_seq)
            aug_labels.append(lab)
            
    return aug_sentences, aug_labels

aug_train_sentences, aug_train_labels = augment_data_imbalance(
    train_sentences, 
    train_labels, 
    distribusi_toko=distribusi, 
    batas_minoritas=50 
)

final_train_sentences = train_sentences + aug_train_sentences
final_train_labels = train_labels + aug_train_labels

print(f"\nTotal Data Latih Awal: {len(train_sentences)}")
print(f"Total Data Latih setelah Target Augmentasi: {len(final_train_sentences)}")


Total Data Latih Awal: 1001
Total Data Latih setelah Target Augmentasi: 5390


In [ ]:
# 3. PEMBUATAN KAMUS DAN PADDING DATA

MAX_LEN = 256

words = set()
tags = set()

for seq in final_train_sentences: 
    for word in seq:
        words.add(word.lower())

for labels in final_train_labels:
    for tag in labels:
        tags.add(tag)

word2idx = {"<PAD>": 0, "<OOV>": 1}
for i, word in enumerate(words):
    word2idx[word] = i + 2
    
idx2word = {i: w for w, i in word2idx.items()}

tags = sorted(list(tags))
if "O" not in tags:
    tags.append("O") 

tag2idx = {t: i for i, t in enumerate(tags)}
idx2tag = {i: t for t, i in tag2idx.items()}

NUM_WORDS = len(word2idx)
NUM_TAGS = len(tag2idx)

print(f"Jumlah Kosakata Unik: {NUM_WORDS}")
print(f"Jumlah Tag (Label) Unik: {NUM_TAGS}")
print("Daftar Tag (Harus sama dengan Android nanti!):", tags)

X_train_enc = [[word2idx.get(w.lower(), word2idx["<OOV>"]) for w in s] for s in final_train_sentences]
Y_train_enc = [[tag2idx[l] for l in s] for s in final_train_labels]

X_test_enc = [[word2idx.get(w.lower(), word2idx["<OOV>"]) for w in s] for s in test_sentences]
Y_test_enc = [[tag2idx[l] for l in s] for s in test_labels]

X_train_pad = pad_sequences(X_train_enc, maxlen=MAX_LEN, padding='post', truncating='post', value=word2idx["<PAD>"])
Y_train_pad = pad_sequences(Y_train_enc, maxlen=MAX_LEN, padding='post', truncating='post', value=tag2idx["O"])

X_test_pad = pad_sequences(X_test_enc, maxlen=MAX_LEN, padding='post', truncating='post', value=word2idx["<PAD>"])
Y_test_pad = pad_sequences(Y_test_enc, maxlen=MAX_LEN, padding='post', truncating='post', value=tag2idx["O"])

Y_train_pad = np.expand_dims(Y_train_pad, -1)
Y_test_pad = np.expand_dims(Y_test_pad, -1)

print(f"Bentuk X_train: {X_train_pad.shape}")
print(f"Bentuk Y_train: {Y_train_pad.shape}")

Jumlah Kosakata Unik: 18556
Jumlah Tag (Label) Unik: 17
Daftar Tag (Harus sama dengan Android nanti!): ['B-diskon', 'B-harga_satuan', 'B-harga_total_barang', 'B-jumlah', 'B-nama_barang', 'B-nama_toko', 'B-tanggal', 'B-total_belanja', 'I-diskon', 'I-harga_satuan', 'I-harga_total_barang', 'I-jumlah', 'I-nama_barang', 'I-nama_toko', 'I-tanggal', 'I-total_belanja', 'O']
Bentuk X_train: (5390, 256)
Bentuk Y_train: (5390, 256, 1)


In [ ]:
# 4. FASTTEXT & EMBEDDING MATRIX

EMBEDDING_DIM = 300 

fasttext_path = "cc.id.300.vec" 

print("Membaca FastText...")
embedding_index = {}
with open(fasttext_path, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.rstrip().rsplit(' ')
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = coefs

print(f"Ditemukan {len(embedding_index)} vektor kata di FastText.")

embedding_matrix = np.zeros((NUM_WORDS, EMBEDDING_DIM))
for word, i in word2idx.items():
    embedding_vector = embedding_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

print(f"Matriks Embedding berhasil dibuat dengan bentuk: {embedding_matrix.shape}")

Membaca FastText...
Ditemukan 2000000 vektor kata di FastText.
Matriks Embedding berhasil dibuat dengan bentuk: (18556, 300)


In [ ]:
# 5. Tuning (Class Weight) // TIDAK TERPAKAI karena pakai CRFModel dari keras-crf tidak mendukung class_weight, jadi diganti dengan augmentasi data untuk mengatasi ketidakseimbangan kelas

labels_flat = [l for seq in Y_encoded for l in seq]
classes = np.unique(labels_flat)

weights = compute_class_weight('balanced', classes=classes, y=labels_flat)
class_weight_dict = dict(zip(classes, weights))

class_weight_dict[0] = 0.0 

sample_weights = np.zeros_like(Y_padded, dtype=np.float32)
for i in range(Y_padded.shape[0]):
    for j in range(Y_padded.shape[1]):
        sample_weights[i, j] = class_weight_dict.get(Y_padded[i, j], 1.0)

print("Matriks Sample Weights siap!")

Matriks Sample Weights siap!


In [ ]:
# 6. MEMBANGUN MODEL BILSTM-CRF

input_layer = Input(shape=(MAX_LEN,))

embed_layer = Embedding(
    input_dim=NUM_WORDS, 
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix], 
    input_length=MAX_LEN, 
    trainable=True, 
    mask_zero=False 
)(input_layer)

drop_layer = Dropout(0.3)(embed_layer)
bilstm_layer = Bidirectional(LSTM(units=128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(drop_layer)

dense_layer = Dense(NUM_TAGS)(bilstm_layer)

crf_layer = CRF(dtype='float32')
output_layer = crf_layer(dense_layer)

base_model = Model(inputs=input_layer, outputs=output_layer)
model = ModelWithCRFLoss(base_model)
model.compile(optimizer='adam', run_eagerly=True) 

base_model.summary()



Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 256)]             0         
                                                                 
 embedding (Embedding)       (None, 256, 300)          5566800   
                                                                 
 dropout (Dropout)           (None, 256, 300)          0         
                                                                 
 bidirectional (Bidirection  (None, 256, 256)          439296    
 al)                                                             
                                                                 
 dense (Dense)               (None, 256, 17)           4369      
                                                                 
 crf (CRF)                   ((None, 256),             289       
                              (None, 256, 17),             

In [ ]:
# Menghapus dimensi ekstra agar menjadi 2D (Batch, Timesteps) sesuai kemauan tf2crf
import numpy as np

if len(Y_train_pad.shape) == 3:
    Y_train_pad = np.squeeze(Y_train_pad, axis=-1)

if len(Y_test_pad.shape) == 3:
    Y_test_pad = np.squeeze(Y_test_pad, axis=-1)

print(f"Bentuk Baru Y_train: {Y_train_pad.shape}") 
print(f"Bentuk Baru Y_test: {Y_test_pad.shape}")   

Bentuk Baru Y_train: (5390, 256)
Bentuk Baru Y_test: (251, 256)


In [ ]:
# 7. TRAINING DENGAN TUNING MAKSIMAL

early_stop = EarlyStopping(
    monitor='val_loss_val', 
    patience=5, 
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss_val', 
    factor=0.5, 
    patience=2, 
    min_lr=0.0001,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_ner_weights.h5', 
    monitor='val_loss_val',
    save_best_only=True,
    save_weights_only=True, 
    verbose=1
)

print("\nMemulai proses training ...")

history = model.fit(
    X_train_pad, Y_train_pad, 
    validation_data=(X_test_pad, Y_test_pad),
    batch_size=16, 
    epochs=50,     
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

print("Pelatihan selesai dan memori terbaik telah dipulihkan!")


Memulai proses training (Mode Irit RAM)...
Epoch 1/50
337/337 [==============================] - ETA: 0s - loss: 87.0339 - accuracy: 0.9029 
Epoch 1: val_loss_val improved from inf to 44.26305, saving model to best_ner_weights.h5
337/337 [==============================] - 6316s 19s/step - loss: 87.0339 - accuracy: 0.9029 - val_loss_val: 44.2630 - val_val_accuracy: 0.9418 - lr: 0.0010
Epoch 2/50
337/337 [==============================] - ETA: 0s - loss: 17.0792 - accuracy: 0.9786 
Epoch 2: val_loss_val improved from 44.26305 to 36.52936, saving model to best_ner_weights.h5
337/337 [==============================] - 5190s 15s/step - loss: 17.0792 - accuracy: 0.9786 - val_loss_val: 36.5294 - val_val_accuracy: 0.9527 - lr: 0.0010
Epoch 3/50
337/337 [==============================] - ETA: 0s - loss: 8.0711 - accuracy: 0.9901 
Epoch 3: val_loss_val improved from 36.52936 to 35.09064, saving model to best_ner_weights.h5
337/337 [==============================] - 6200s 18s/step - loss: 8.0711

In [ ]:
# 8. EVALUASI & CLASSIFICATION REPORT

print("Mengevaluasi model pada data ujian...")
y_pred = model.predict(X_test_pad)

y_pred_flat = []
y_true_flat = []

for i in range(len(y_pred)):
    for j in range(MAX_LEN):
        if X_test_pad[i][j] != word2idx["<PAD>"]:
            y_pred_flat.append(idx2tag[y_pred[i][j]])
            true_val = Y_test_pad[i][j][0] if isinstance(Y_test_pad[i][j], (list, np.ndarray)) else Y_test_pad[i][j]
            y_true_flat.append(idx2tag[true_val])
            
print(classification_report(y_true_flat, y_pred_flat))

Mengevaluasi model pada data ujian...
8/8 [==============================] - 42s 5s/step
                      precision    recall  f1-score   support

            B-diskon       0.80      0.69      0.74        96
      B-harga_satuan       0.83      0.87      0.85       574
B-harga_total_barang       0.84      0.86      0.85       835
            B-jumlah       0.84      0.90      0.87       561
       B-nama_barang       0.80      0.76      0.78       892
         B-nama_toko       0.73      0.69      0.71       202
           B-tanggal       0.90      0.94      0.92       216
     B-total_belanja       0.78      0.81      0.80       246
            I-diskon       0.80      0.80      0.80       166
      I-harga_satuan       0.82      0.88      0.85       629
I-harga_total_barang       0.85      0.90      0.88      1551
            I-jumlah       0.75      0.25      0.38        24
       I-nama_barang       0.88      0.85      0.86      3290
         I-nama_toko       0.66      0.65 

In [ ]:
import json

# 9. EKSPOR KAMUS KATA DAN LABEL KE JSON

with open('word2idx.json', 'w', encoding='utf-8') as f:
    json.dump(word2idx, f)

idx2label_str_keys = {str(k): v for k, v in idx2tag.items()}
with open('idx2label.json', 'w', encoding='utf-8') as f: 
    json.dump(idx2label_str_keys, f)

print("File word2idx.json dan idx2label.json berhasil dibuat!")

✅ File word2idx.json dan idx2label.json berhasil dibuat!


In [ ]:
import tensorflow as tf
import json

# 10. KONVERSI KE TFLITE & EKSPOR MATRIKS CRF

print("1. Mengonversi model ke TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(base_model) 

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False

converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('model_bilstm.tflite', 'wb') as f:
    f.write(tflite_model)
print("Model TFLite (model_bilstm.tflite) berhasil disimpan!")

print("2. Mengekstrak matriks transisi CRF...")
try:
    crf_transition = crf_layer.transitions.numpy().tolist()

    with open('crf_transitions.json', 'w', encoding='utf-8') as f:
        json.dump(crf_transition, f)
        
    print("File crf_transitions.json berhasil dibuat!")
except Exception as e:
    print("Gagal mengekstrak matriks transisi, error:", e)

1. Mengonversi model ke TFLite...
INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmpz72fral_\assets


INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmpz72fral_\assets


✅ Model TFLite (model_bilstm.tflite) berhasil disimpan!
2. Mengekstrak matriks transisi CRF...
✅ File crf_transitions.json berhasil dibuat!
--- 🎉 SEMUA PROSES SELESAI 🎉 ---


In [ ]:
import tensorflow as tf

# Buat model yang hanya sampai Dense Layer (tanpa CRF)
model_export = Model(inputs=input_layer, outputs=dense_layer)

print("Mengonversi model ke TFLite (Hanya sampai Dense Layer)...")
converter = tf.lite.TFLiteConverter.from_keras_model(model_export) 

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False

converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('model_bilstm.tflite', 'wb') as f:
    f.write(tflite_model)

Mengonversi model ke TFLite (Hanya sampai Dense Layer)...
INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmp0lhblti3\assets


INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmp0lhblti3\assets


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout

print("1. Mengunci Dimensi (Static Shape)...")
input_static = Input(batch_shape=(1, MAX_LEN))

embed_static = Embedding(NUM_WORDS, EMBEDDING_DIM, mask_zero=False)(input_static)
drop_static = Dropout(0.3)(embed_static)
bilstm_static = Bidirectional(LSTM(units=128, return_sequences=True))(drop_static)
dense_static = Dense(NUM_TAGS)(bilstm_static)

model_export = Model(inputs=input_static, outputs=dense_static)

print("2. Mentransfer kepintaran AI (Tanpa Latih Ulang)...")
for i in range(len(model_export.layers)):
    model_export.layers[i].set_weights(base_model.layers[i].get_weights())

print("3. Mengekspor ke TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model_export) 
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open('model_bilstm.tflite', 'wb') as f:
    f.write(tflite_model)
    
print("Model static berhasil dibuat!")

1. Mengunci Dimensi (Static Shape)...
2. Mentransfer kepintaran AI (Tanpa Latih Ulang)...
3. Mengekspor ke TFLite...
INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmp_b0qjxmw\assets


INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmp_b0qjxmw\assets


✅ Model Static Berhasil Dibuat! Silakan Download.
